# Create xyz and CIF files of Johnson NPs of different sizes and forms from a dataset of compounds (CIF dataset)

#### Two classes : one for the fcc trigonal bipyramid and one for pentagonal bipyramids (decahedron) and elongated bipyramids

## Imports

In [1]:
import os
import sys
import tkinter

print(os.getcwd())
cwd0 = './styles/'
sys.path.append(cwd0)
from pathlib import Path
import numpy as np
import visualID as vID
from visualID import  fg, hl, bg

from pyNanoMatBuilder import johnsonNPs as jNP
from pyNanoMatBuilder import utils as pyNMBu
from pyNanoMatBuilder import data
import importlib

from ase import io
from ase.atoms import Atoms
from ase.geometry import cellpar_to_cell
from ase.spacegroup import get_spacegroup
from ase.io import read
from ase.io import write
from ase.visualize import view

from debyecalculator import DebyeCalculator
import pyNanoMatBuilder.utilsDC as pyNMBuDC

/home/sara/pyNanoMatBuilder


## 1. Class for fcc trigonal bipyramids

#### NB: the Johnson class is instantiated by defining a compound and its interatomic distance. The function FindInterAtomicDist, which is defined in utils, calculates the interatomic distance by extracting the cell paratemers and uses well known formulas for fcc, bcc and hcp. Thus, compounds with unknown interatomic distances are not used to create Platonic NPs.

<span style="color:#FF0000"> Problem : not possible to create the forms for multiple elements like TiO2 or NaCl


### Definition of the 'Johnson_fcctbp_Files' class

In [2]:
class Johnson_fcctbp_Files :
    """
    A class for generating XYZ and CIF files of fcc trigonal bipyramid (fcctbp) Johnson nanoparticles (NPs) 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data and sizes.
            Args:
        path (str) : Path that will contain the created xyz files
        cif data (DataFrame):  DataFrame containing CIF files of different compounds
        sizes (array-like): Array of the number of bonds per edges, ex: [2,3,4]
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): If False, details of the files are printed.
            Methods:
        create_johnson_fcctbp(noOutput, path): Generates fcctbp Johnson nanoparticlesand saves their files.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # Dictionary to store loaded CIF files
        self.sizes= sizes # List of sizes 

        
        self.create_johnson_fcctbp(noOutput,max_size,path) # Call method to generate Johnson nanoparticles
  
    def create_johnson_fcctbp(self,noOutput,max_size,path):
        """
        Genenrate platonic NPs and their files from cif data.
        Args:
            noOutput (bool): If False, prints details about the process.
            path (str): Path where xyz/cif files will be created.
        Note:
        - The nanoparticles are created using classes of the module 'jNP' 
        and the class 'fcctbp'
        """
    
        
        for cif_name, cif_file in self.cif_data['cif file'].items():
            
            # Load cif informations
            element=cif_name.split()[0] # Extract chemical element from CIF name
            
            if not (element=='TiO2' or element=='NaCl'): # Skip TiO2 and NaCl since they are not processed
                self.cif_name=cif_name
                
                # Extract the structure name for the name of the files, for example 'Rutile' or 'Anatase' or 'Alpha'
                if len(self.cif_name.split())==2 : 
                    structure=self.cif_name.split()[1]
                else : 
                    structure=None
                
                if not noOutput :
                    print(f'\n\033[1m{cif_name.center(50)}\033[0m\n')
                cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
                    
                # Determine forms based on crystal type
                if self.crystal_type=='fcc' :
                    dist= pyNMBu.FindInterAtomicDist(self)# Extract the interatomic distance
                    index=0 
                    if not noOutput :
                        print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type}. \033[0m ")
   
                    # Create instances for each form and size
                    for i in self.sizes :
                        index += 1 
                        if not noOutput :
                            print(f'Number of bonds is {i}')
                        TestNP =jNP.fcctbp(
                            element=element,
                            Rnn=dist,
                            nLayerTd=i,
                            postAnalyzis=True,
                            aseView=False,
                            thresholdCoreSurface=1,
                            skipSymmetryAnalyzis=True,
                            noOutput= True
                        )

                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                            pyNMBu.writexyz_generalized_johnson(self,structure,path,TestNP,index,noOutput) 
                        else :
                            break
                

                else : 
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} xyz/cif files can't be created for {self.cif_name} because the crystal type isn't fcc.\033[0m  ")   
                    pass

          
            else :
                pass          

In [3]:
Johnson_fcctbp_Files?

Init signature:
Johnson_fcctbp_Files(
    path,
    cif_data,
    sizes,
    max_size: float = 50,
    noOutput: bool = True,
)
Docstring:     
A class for generating XYZ and CIF files of fcc trigonal bipyramid (fcctbp) Johnson nanoparticles (NPs) 
from a dataset of compounds (CIF dataset). This process enables the creation 
of a well-structured database optimized for machine learning applications, 
ensuring consistency in format and data representation.
Init docstring:
Initialize the class with CIF data and sizes.
    Args:
path (str) : Path that will contain the created xyz files
cif data (DataFrame):  DataFrame containing CIF files of different compounds
sizes (array-like): Array of the number of bonds per edges, ex: [2,3,4]
max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
noOutput (bool): If False, details of the files are printed.
    Methods:
create_johnson_fcctbp(noOutput, path): Generates fcctbp 

### Example of use of the  Johnson_fcctbp_Files class

##### Create all the possible forms from a dataset of CIF files

If noOutput=False :  
 - <span style="color:#008000"> in green : the allowed shapes created (fcc) </span> 
 - <span style="color:#FF0000">in red : the shapes that are not created (not fcc)</span>

In [4]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/fcctbp_johnson_1_5nm'

# Sizes of the NPs
sizes=np.arange(1,30,1) # Control the sizes by the number of bonds per edges : here 1 to 25 bonds (step of 1)

# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the maximum number of bonds defined, here 25.
class_test=Johnson_fcctbp_Files(path,cif_data=data.pyNMBcif.CIFdf,sizes=sizes,max_size=5,noOutput= False) # Ex: the maximal size is a circumsphere diameter of 5 nm.



print(t.chrono_stop(hdelay=True))


                      Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
  xyz/cif files can't be created for Fe bcc because the crystal type isn't fcc.  

                     Mn alpha                     

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9011068-Mn_alpha.cif
  xyz/cif files can't be created for Mn alpha because the crystal type isn't fcc.  

                     Mn beta                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod1539039-Mn_beta.cif
  xyz/cif files can't be created for Mn beta because the crystal type isn't fcc.  

                      Co hcp                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9008492-Co_hcp.cif
  xyz/cif files can't be created for Co hcp because the crystal type isn't fcc.  

                      Co fcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_databas

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(
/home/sara/pyNanoMatBuilder/pyNanoMatBuilder/platonicNPs.py:918: RuntimeWarning: invalid value encountered in scalar divide
  return self.heightOfPyramid()/(self.nLayer-1)


Circumscribed sphere diameter =4.096877510169587
  xyz file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000010_0000000.xyz 
  cif file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000010_0000000.cif
Number of bonds is 11
Circumscribed sphere diameter =4.506565261186544
  xyz file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000011_0000000.xyz 
  cif file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000011_0000000.cif
Number of bonds is 12
Circumscribed sphere diameter =4.916253012203502
  xyz file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000012_0000000.xyz 
  cif file created:database_1_5nm/fcctbp_johnson_1_5nm/Co_fcc_fcctbp_0000012_0000000.cif
Number of bonds is 13

                    Co epsilon                    

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod9012884-Co_epsilon.cif
  xyz/cif files can't be created for Co epsilon because the crystal type isn't fcc.  

                     

## Class for pentagonal bipyramids (decahedron) and elongated bipyramids

### Definition of the 'Johnson_epbpyM_Files' class

In [5]:
class Johnson_epbpyM_Files:### Pentagonal bipyramids (decahedron) and elongated bipyramids
    """
    A class for generating XYZ and CIF files of pentagonal bipyramids (decahedron) and elongated bipyramids Johnson nanoparticles (NPs) 
    from a dataset of compounds (CIF dataset). This process enables the creation 
    of a well-structured database optimized for machine learning applications, 
    ensuring consistency in format and data representation.
    """
    def __init__(self, path, cif_data,sizes,max_size: float=50,noOutput:bool = True):
        """
        Initialize the class with CIF data and sizes.
            Args:
        path (str) : Path that will contain the created xyz files
        cif data (DataFrame):  DataFrame containing CIF files of different compounds
        sizes (array-like): Array of the number of bonds per edges, ex: [2,3,4]
        max_size (float, optional) : Maximal size for the NPs, equals to the diameter of the circumscribed sphere, equals 50 nm by default.
        noOutput (bool): If False, details of the files are printed.
            Methods:
        create_johnson_epbpyM(noOutput, path): Generates fcctbp Johnson nanoparticlesand saves their files.
        """
        self.cif_data = cif_data  # DataFrame containing CIF
        self.loaded_cifs = {}  # Dictionary to store loaded CIF files
        self.sizes=sizes # [sizeP,sizeE,Marks]
        self.sizeP= np.unique(sizes [:, 0])# list of the number of bonds on the edge of the pentagonal cross section
        self.sizeE= np.unique(sizes [:, 1])# list of the number of bonds on the edge of the elongated part
        self.Marks= np.unique(sizes [:, 2]) # list of numbers of truncated atoms

        
        self.create_johnson_epbpyM(noOutput,sizes,max_size,path) # Call method to generate Johnson nanoparticles
  
    def create_johnson_epbpyM(self,noOutput,sizes,max_size,path):
        """
        Genenrate platonic NPs and their files from cif data.
        Args:
            noOutput (bool): If False, prints details about the process.
            path (str): Path where xyz/cif files will be created.
        Note:
        - The nanoparticles are created using classes of the module 'jNP' 
        and the class 'fcctbp'
        """
    
        if not noOutput:
            print(f' The sizes are given using the following format : [number of bonds per edge of pentagonal cross section, number of bonds per edge of elongated part, number of truncated atoms]')
        for cif_name, cif_file in self.cif_data['cif file'].items():
            
            # Load cif informations
            element=cif_name.split()[0] # Extract chemical element from CIF name
            
            if not (element=='TiO2' or element=='NaCl'): # Skip TiO2 and NaCl since they are not processed
                self.cif_name=cif_name
                
                # Extract the structure name for the name of the files, for example 'Rutile' or 'Anatase' or 'Alpha'
                if len(self.cif_name.split())==2 : 
                    structure=self.cif_name.split()[1]
                else : 
                    structure=None
                
                if not noOutput :
                    print(f'\n\033[1m{cif_name.center(50)}\033[0m\n')
                cif_info = pyNMBu.load_cif(self,cif_file,noOutput)
                    
                # Determine forms based on crystal type
                if self.crystal_type=='fcc' :
                    dist= pyNMBu.FindInterAtomicDist(self)# Extract the interatomic distance
                elif self.crystal_type=='bcc':
                    dist= pyNMBu.FindInterAtomicDist(self) # Extract the interatomic distance
                elif self.crystal_type=='hcp':
                    dist= pyNMBu.FindInterAtomicDist(self) # Extract the interatomic distance
                else : 
                    dist=None
                    if not noOutput :
                        print(f" {bg.LIGHTREDB} xyz/cif files can't be created for {self.cif_name} because the interatomic distance is unknown.\033[0m  ")   
                        
                if not dist==None:
 
                    index=0 
                    if not noOutput :
                        print(f" {bg.LIGHTGREENB} xyz/cif files can be created for {self.cif_name} of Bravais lattice={self.crystal_type}. \033[0m ")
   
                    # Create instances for each form and size
                    # for i in self.sizeP :
                    #     for y in self.sizeE:
                    #         for z in self.Marks:

                    for i, y, z in sizes:
                        index += 1 
                        if not noOutput :
                            print(f'{bg.LIGHTBLUEB} [{i},{y},{z}]  ')
                        TestNP =jNP.epbpyM(
                            element=element,
                            Rnn=dist,
                            sizeP=i,
                            sizeE=y,
                            Marks=z,
                            postAnalyzis=True,
                            aseView=False,
                            thresholdCoreSurface=1,
                            skipSymmetryAnalyzis=True,
                            noOutput= True
                        )
           
                        circumsphere_diameter=TestNP.radiusCircumscribedSphere*2*0.1 # Setting a maximal size of NPs : circumscribed sphere diameter
                        if circumsphere_diameter<max_size :
                            if not noOutput :
                                print(f'{bg.LIGHTBLUEB}Circumscribed sphere diameter ={circumsphere_diameter}\033[0m')
                            pyNMBu.writexyz_generalized_johnson(self,structure,path,TestNP,index,noOutput) 
                        else :
                            break
                

          
            else :
                pass          

### Example of use of the Johnson_epbpyM_Files class

An example of possible combinations of sizes (size of the pentagonal cap, size of the elongated part and Marks=truncated atoms per vertices). 

In [6]:

import numpy as np
from itertools import product

sizeP = np.arange(3, 7, 1)  # Ex: [3, 4, 5, 6]
sizeE = np.arange(3, 5, 1)  # Ex: [3, 4, ..., 14]
Marks = np.arange(0, 4, 1)  # Ex: [0, 1, 2, 3]

# Générer toutes les combinaisons possibles
valid_combinations = [(p, e, m) for p, e, m in product(sizeP, sizeE, Marks) if m <= (p-1) // 2]

# Convertir en tableau numpy si nécessaire
valid_combinations = np.array(valid_combinations)

print(valid_combinations)



[[3 3 0]
 [3 3 1]
 [3 4 0]
 [3 4 1]
 [4 3 0]
 [4 3 1]
 [4 4 0]
 [4 4 1]
 [5 3 0]
 [5 3 1]
 [5 3 2]
 [5 4 0]
 [5 4 1]
 [5 4 2]
 [6 3 0]
 [6 3 1]
 [6 3 2]
 [6 4 0]
 [6 4 1]
 [6 4 2]]


In [7]:
t = pyNMBu.timer()
t.chrono_start()

# Path where the files are created
path='database_1_5nm/epbpyM_johnson_1_5nm'

# Possiblity to define a maximal size: max_size (a circumsphere diameter). If max_size isn't defined, the NPs will have the maximum number of bonds defined, here 25.
class_test=Johnson_epbpyM_Files(path,cif_data=data.pyNMBcif.CIFdf,sizes=valid_combinations,max_size=5,noOutput= False) # Ex: the maximal size is a circumsphere diameter of 5 nm.



print(t.chrono_stop(hdelay=True))

 The sizes are given using the following format : [number of bonds per edge of pentagonal cross section, number of bonds per edge of elongated part, number of truncated atoms]

                      Fe bcc                      

Absolute path to CIF: /home/sara/pyNanoMatBuilder/cif_database/cod5000217-Fe_bcc.cif
  xyz/cif files can be created for Fe bcc of Bravais lattice=bcc.  
 [3,3,0]  
Circumscribed sphere diameter =1.563014668022719
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000001_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000001_0000000.cif
 [3,3,1]  
Circumscribed sphere diameter =1.563014668022719
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000002_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000002_0000000.cif
 [3,4,0]  
Circumscribed sphere diameter =1.8235171126931722
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000003_00

/home/sara/Python3/Debye_calc/lib/python3.11/site-packages/ase/io/cif.py:408: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(229, setting=1). This may result in wrong setting!
  warnings.warn(


 [4,4,0]  
Circumscribed sphere diameter =2.084019557363625
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000007_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000007_0000000.cif
 [4,4,1]  
Circumscribed sphere diameter =2.084019557363625
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000008_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000008_0000000.cif
 [5,3,0]  
Circumscribed sphere diameter =2.247742890352782
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000009_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000009_0000000.cif
 [5,3,1]  
Circumscribed sphere diameter =2.084019557363625
  xyz file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000010_0000000.xyz 
  cif file created:database_1_5nm/epbpyM_johnson_1_5nm/Fe_bcc_epbpyM_0000010_0000000.cif
 [5,3,2]  
Circumscribed sphere diameter =2.